In [35]:
# imports
import pandas as pd
from Bio.PDB import PDBParser
import os
import glob
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import jaccard_score
import lightgbm as lgb
from collections import defaultdict, deque

In [36]:
# Amino acids constants (renaming from triple to single digits)

AA3_TO_AA1 = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C",
    "GLN": "Q", "GLU": "E", "GLY": "G", "HIS": "H", "ILE": "I",
    "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P",
    "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V"
}

# feauture name constants

HYDROPHOBIC = {"A", "V", "I", "L", "M", "F", "W", "Y", "P"}
POLAR = {"S", "T", "N", "Q", "C", "G"}
POSITIVE = {"K", "R", "H"}
NEGATIVE = {"D", "E"}
AROMATIC = {"F", "W", "Y", "H"}

In [37]:
#convert label strings to sets

def parse_binding_string(s):
    """
    Converts a string like
    'A_24 A_25 A_26'
    into a set like thiiiis:
    {'A_24', 'A_25', 'A_26'}
    """
    if pd.isna(s) or not str(s).strip():
        return set()
    return set(str(s).strip().split())


def load_train_labels(csv_path):
    """
    Loads train.csv and creates the extra column 'binding_set'.
    
    Expected format:
    id,residue
    0,A_24 A_25 A_26 ...
    1,A_14 A_15 A_16 ...
    """
    df = pd.read_csv(csv_path)

    # ID as string, so '0' is cleanly matched to filename
    df["id"] = df["id"].astype(str)

    # convert pocket-residues to sets
    df["binding_set"] = df["resid"].apply(parse_binding_string)

    return df

In [38]:
# helper function for filenames

def extract_pdb_id(filepath):
    """
    Extracts from filenames:
    '0_protein.pdb'
    the ID:
    '0'
    """
    base = os.path.basename(filepath) # eg. '0_protein.pdb'
    name = os.path.splitext(base)[0] # eg. '0_protein'
    pdb_id = name.split("_")[0] # eg. '0'
    return pdb_id

In [39]:
# helper function for residue keys

def make_residue_key(chain_id, residue):
    """
    buiolds a residue key in the same format like in the .csv:
    - A_24
    - A_123_A
    
    residue.id is typically:
    (' ', 24, ' ')
    or
    (' ', 123, 'A')
    """
    resid = residue.id[1]
    icode = residue.id[2].strip()

    if icode:
        return f"{chain_id}_{resid}_{icode}"
    return f"{chain_id}_{resid}"

In [40]:
# pasre single pdb file helper function

def parse_pdb_file(pdb_path, pdb_id=None):
    """
    parses a single pdb file and returns a list of dictionaries.
    only standard amino acids with a CA-atom are taken.
    """
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_id or "protein", pdb_path)

    rows = []

    # Only use first model
    model = next(structure.get_models())

    for chain in model:
        for residue in chain:
            # only standard amino acids
            if residue.resname not in AA3_TO_AA1:
                continue

            # only residues with alpha carbon
            if "CA" not in residue:
                continue

            ca = residue["CA"].get_coord()

            row = {
                "pdb_id": str(pdb_id) if pdb_id is not None else None,
                "chain_id": chain.id,
                "residue_key": make_residue_key(chain.id, residue),
                "resname3": residue.resname,
                "aa": AA3_TO_AA1[residue.resname],
                "resid": residue.id[1],
                "icode": residue.id[2].strip(),
                "x": float(ca[0]),
                "y": float(ca[1]),
                "z": float(ca[2]),
            }

            rows.append(row)

    return rows

In [41]:
# parse all pdb files

def parse_all_pdbs(pdb_dir, max_files=None):
    """
    parses all .pdb files from one folder.
    
    Expected filnames:
    0_protein.pdb
    1_protein.pdb
    2_protein.pdb
    ...
    
    return a dataframe with all residues.
    """
    pdb_files = sorted(glob.glob(os.path.join(pdb_dir, "*.pdb")))
    all_rows = []

    if max_files is not None:
        pdb_files = pdb_files[:max_files]

    print(f"found pdb files: {len(pdb_files)}")

    for i, pdb_file in enumerate(pdb_files, start=1):
        pdb_id = extract_pdb_id(pdb_file)

        try:
            rows = parse_pdb_file(pdb_file, pdb_id=pdb_id)
            all_rows.extend(rows)
        except Exception as e:
            print(f"[FEHLER] {pdb_file}: {e}")

        if i % 100 == 0 or i == len(pdb_files):
            print(f"Verarbeitet: {i}/{len(pdb_files)}")

    df = pd.DataFrame(all_rows)
    return df

In [ ]:
# build train dataframe better version

def build_train_dataframe(pdb_dir, labels_df, max_files=None, drop_bad_proteins=False,
                          max_missing_fraction=0.30, max_missing_abs=5):
    """
    builds a training dataframe on residue level.

    every row = one residue
    target = 1, if residue is in binding pocket
    target = 0 otherwise

    Additionally:
    - only concerns label residues tha are actually in the pdb
    - create a report for missing label residues
    - is able to exclude problematic label residues optionally
    """
    pdb_files = sorted(glob.glob(os.path.join(pdb_dir, "*.pdb")))

    if max_files is not None:
        pdb_files = pdb_files[:max_files]

    binding_map = dict(zip(labels_df["id"], labels_df["binding_set"]))

    all_rows = []
    missing_reports = []

    kept_proteins = 0
    dropped_proteins = 0

    for i, pdb_file in enumerate(pdb_files, start=1):
        pdb_id = extract_pdb_id(pdb_file)

        try:
            rows = parse_pdb_file(pdb_file, pdb_id=pdb_id)
        except Exception as e:
            print(f"[FEHLER] {pdb_file}: {e}")
            continue

        binding_set = binding_map.get(pdb_id, set())
        parsed_keys = {row["residue_key"] for row in rows}

        # only keep labels that are actually existing in the structure
        effective_binding_set = binding_set & parsed_keys
        missing_labels = sorted(binding_set - parsed_keys)

        n_labels = len(binding_set)
        n_missing = len(missing_labels)
        missing_fraction = (n_missing / n_labels) if n_labels > 0 else 0.0

        # Heuristic: protein is problemetic if too many label residues are missing
        is_problematic = (
            (n_missing > max_missing_abs) or
            (missing_fraction > max_missing_fraction)
        )

        missing_reports.append({
            "pdb_id": pdb_id,
            "n_labels": n_labels,
            "n_parsed_residues": len(parsed_keys),
            "n_effective_labels": len(effective_binding_set),
            "n_missing_labels": n_missing,
            "missing_fraction": missing_fraction,
            "is_problematic": int(is_problematic),
            "missing_labels": " ".join(missing_labels)
        })

        # optional: problematic proteins are skipped
        if drop_bad_proteins and is_problematic:
            dropped_proteins += 1
            if i % 100 == 0 or i == len(pdb_files):
                print(f"Training files processed: {i}/{len(pdb_files)}")
            continue

        for row in rows:
            row["target"] = int(row["residue_key"] in effective_binding_set)
            row["n_labels_total"] = n_labels
            row["n_missing_labels"] = n_missing
            row["missing_fraction"] = missing_fraction
            row["is_problematic_protein"] = int(is_problematic)
            all_rows.append(row)

        kept_proteins += 1

        if i % 100 == 0 or i == len(pdb_files):
            print(f"Training files processed: {i}/{len(pdb_files)}")

    train_df = pd.DataFrame(all_rows)
    missing_df = pd.DataFrame(missing_reports)

    print()
    print("Build-Train-DataFrame Zusammenfassung:")
    print(f"Behaltene Proteine: {kept_proteins}")
    print(f"Ausgeschlossene Proteine: {dropped_proteins}")
    print(f"Proteine gesamt im Report: {len(missing_df)}")

    return train_df, missing_df

In [43]:
# build test dataframe

def build_test_dataframe(pdb_dir, max_files=None):
    """
    builds a test dataframe on residue level.
    yet without target.
    """
    test_df = parse_all_pdbs(pdb_dir, max_files=max_files)
    return test_df

In [44]:
# feature engineering (gathering info which i think is relevant)

def add_basic_aa_features(df):
    """
    adds simple properties of the amino acids.
    """
    df = df.copy()

    df["is_hydrophobic"] = df["aa"].isin(HYDROPHOBIC).astype(int)
    df["is_polar"] = df["aa"].isin(POLAR).astype(int)
    df["is_positive"] = df["aa"].isin(POSITIVE).astype(int)
    df["is_negative"] = df["aa"].isin(NEGATIVE).astype(int)
    df["is_aromatic"] = df["aa"].isin(AROMATIC).astype(int)

    return df


def safe_mean_topk(sorted_distances, k):
    """
    mean distance of the nearest k neighbours.
    if there are less than k neighbours, an adaption is made
    """
    k = min(k, sorted_distances.shape[1])
    if k == 0:
        return np.zeros(sorted_distances.shape[0], dtype=float)
    return sorted_distances[:, :k].mean(axis=1)


def add_geometric_features(df):
    """
    calculates geometric properties per protein based on the CA-coordinates.
    """
    all_groups = []

    for pdb_id, g in df.groupby("pdb_id", sort=False):
        g = g.copy().reset_index(drop=True)

        coords = g[["x", "y", "z"]].values

        # protein center
        center = coords.mean(axis=0)
        g["dist_to_center"] = np.linalg.norm(coords - center, axis=1)

        dist_mean = g["dist_to_center"].mean()
        dist_std = g["dist_to_center"].std() + 1e-8
        g["dist_to_center_z"] = (g["dist_to_center"] - dist_mean) / dist_std

        # pairwise distance matrix
        dmat = np.linalg.norm(coords[:, None, :] - coords[None, :, :], axis=-1)

        # self distance is ignored
        np.fill_diagonal(dmat, np.inf)

        # number of neighbours within different radii
        g["n_neighbors_4"] = (dmat < 4.0).sum(axis=1) 
        g["n_neighbors_6"] = (dmat < 6.0).sum(axis=1)
        g["n_neighbors_8"] = (dmat < 8.0).sum(axis=1)
        g["n_neighbors_10"] = (dmat < 10.0).sum(axis=1)
        g["n_neighbors_12"] = (dmat < 12.0).sum(axis=1) 

        # distance to the nearest neighbours
        sorted_d = np.sort(dmat, axis=1)
        g["mean_knn_3"] = safe_mean_topk(sorted_d, 3)
        g["mean_knn_5"] = safe_mean_topk(sorted_d, 5)
        g["mean_knn_10"] = safe_mean_topk(sorted_d, 10)

        g["min_knn_1"] = sorted_d[:, 0] 
        g["std_knn_5"] = np.std(sorted_d[:, :5], axis=1) 
        g["std_knn_10"] = np.std(sorted_d[:, :10], axis=1) 

        frac_hydrophobic = []
        frac_polar = []
        frac_positive = []
        frac_negative = []
        frac_aromatic = []

        aas = g["aa"].tolist()

        for i in range(len(g)):
            nbr_idx = np.where(dmat[i] < 8.0)[0]
            nbr_idx = nbr_idx[nbr_idx != i]

            if len(nbr_idx) == 0:
                frac_hydrophobic.append(0.0)
                frac_polar.append(0.0)
                frac_positive.append(0.0)
                frac_negative.append(0.0)
                frac_aromatic.append(0.0)
                continue

            nbr_aas = [aas[j] for j in nbr_idx]
            n = len(nbr_aas)

            frac_hydrophobic.append(sum(a in HYDROPHOBIC for a in nbr_aas) / n)
            frac_polar.append(sum(a in POLAR for a in nbr_aas) / n)
            frac_positive.append(sum(a in POSITIVE for a in nbr_aas) / n)
            frac_negative.append(sum(a in NEGATIVE for a in nbr_aas) / n)
            frac_aromatic.append(sum(a in AROMATIC for a in nbr_aas) / n)

        g["frac_hydrophobic_nbrs_8"] = frac_hydrophobic
        g["frac_polar_nbrs_8"] = frac_polar
        g["frac_positive_nbrs_8"] = frac_positive
        g["frac_negative_nbrs_8"] = frac_negative
        g["frac_aromatic_nbrs_8"] = frac_aromatic

        all_groups.append(g)

    return pd.concat(all_groups, ignore_index=True)

In [45]:
# model training

def train_lgbm(train_df, feature_cols, n_splits=5):
    """
    Trains a LightGBM model on residue-level features using GroupKFold cross-validation.

    Each row in train_df corresponds to one residue.
    The function trains one model per fold, using pdb_id as grouping variable
    so that residues from the same protein never appear in both train and validation split.

    Returns:
    - models: list of trained LightGBM models
    - train_df: copy of the input DataFrame with an additional column 'pred'
      containing out-of-fold prediction scores
    """
    X = train_df[feature_cols]
    y = train_df["target"]
    groups = train_df["pdb_id"]

    gkf = GroupKFold(n_splits=n_splits)

    oof_preds = np.zeros(len(train_df))
    models = []

    for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        print(f"\n=== Fold {fold} ===")

        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = lgb.LGBMClassifier(
            n_estimators=800,
            learning_rate=0.03,
            num_leaves=127,
            min_child_samples=50,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.5,
            reg_lambda=1.0,
            class_weight="balanced",
            random_state=42 + fold
        )

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            eval_metric="binary_logloss",
            categorical_feature=["aa", "chain_id"]
        )

        preds = model.predict_proba(X_val)[:, 1]
        oof_preds[val_idx] = preds
        models.append(model)

    train_df = train_df.copy()
    train_df["pred"] = oof_preds

    return models, train_df

In [46]:
# official metric / submission helpers

def find_best_threshold_official_with_postprocessing(
    df,
    thresholds=None,
    distance_threshold=8.0,
    min_component_size=3
):
    """
    Searches for the best classification threshold based on the official Jaccard metric
    after applying spatial postprocessing.

    For each threshold, residues with prediction score above the threshold are selected,
    filtered by keep_largest_spatial_component, and then evaluated with the official-style
    residue-level Jaccard score.

    Returns:
    - best_thr: threshold with the highest score
    - best_score: best official Jaccard score found
    """
    if thresholds is None:
        thresholds = np.linspace(0.05, 0.95, 30)

    best_thr = None
    best_score = -1.0

    for t in thresholds:
        score = evaluate_threshold_official_with_postprocessing(
            df,
            threshold=t,
            distance_threshold=distance_threshold,
            min_component_size=min_component_size
        )

        print(
            f"t={t:.2f} | "
            f"dist={distance_threshold:.1f} | "
            f"min_comp={min_component_size} -> "
            f"Official Jaccard={score:.4f}"
        )

        if score > best_score:
            best_score = score
            best_thr = t

    print("\nBEST THRESHOLD WITH POSTPROCESSING:")
    print(
        f"threshold={best_thr:.2f}, "
        f"distance_threshold={distance_threshold:.1f}, "
        f"min_component_size={min_component_size}, "
        f"Official Jaccard={best_score:.4f}"
    )

    return best_thr, best_score


def evaluate_threshold_official_with_postprocessing(df, threshold, distance_threshold=8.0, min_component_size=3):
    """
    Evaluated using the same global/official logic as your official score,
    but applies spatial post-processing to each protein beforehand.
    """
    pred_rows = []

    for pdb_id, g in df.groupby("pdb_id"):
        pred_res = g.loc[g["pred"] >= threshold, "residue_key"].tolist()

        if len(pred_res) == 0:
            top_res = g.sort_values("pred", ascending=False).iloc[0]["residue_key"]
            pred_res = [top_res]

        pred_res = keep_largest_spatial_component(
            g,
            pred_res,
            distance_threshold=distance_threshold,
            min_component_size=min_component_size
        )

        for r in pred_res:
            pred_rows.append((str(pdb_id), r, 1))

    df_pred = pd.DataFrame(pred_rows, columns=["id", "residue_id", "prediction"])

    # df has to include long formatted targets protein-wise for this function:
    # id, residue_id, true
    target_rows = []
    for pdb_id, g in df.groupby("pdb_id"):
        true_res = set(g.loc[g["target"] == 1, "residue_key"])
        all_res = set(g["residue_key"])
        for r in all_res:
            target_rows.append((str(pdb_id), r, int(r in true_res)))

    df_target = pd.DataFrame(target_rows, columns=["id", "residue_id", "true"])

    df_eval = pd.merge(df_target, df_pred, on=["id", "residue_id"], how="left")
    df_eval["prediction"] = df_eval["prediction"].fillna(0)

    return jaccard_score(df_eval["true"], df_eval["prediction"])


In [47]:
# build submission

def build_submission(test_df, threshold, output_path,
                     distance_threshold=8.0, min_component_size=3):
    """
    Builds a submission file in the required challenge format.

    Expected output format:
    id,prediction

    The prediction column contains a space-separated list of predicted residue keys
    (for example: 'A_24 A_25 B_10'). For each protein, residues are selected by threshold,
    optionally reduced to the largest spatial component, and written to a CSV file.
    """
    rows = []

    for pdb_id, g in test_df.groupby("pdb_id"):
        pred_res = g.loc[g["pred"] >= threshold, "residue_key"].tolist()

        if len(pred_res) == 0:
            top_res = g.sort_values("pred", ascending=False).iloc[0]["residue_key"]
            pred_res = [top_res]

        pred_res = keep_largest_spatial_component(
            g,
            pred_res,
            distance_threshold=distance_threshold,
            min_component_size=min_component_size
        )

        rows.append({
            "id": str(pdb_id),
            "prediction": " ".join(pred_res)
        })

    submission_df = pd.DataFrame(rows)
    submission_df.to_csv(output_path, index=False)
    return submission_df

In [48]:
def keep_largest_spatial_component(g, residue_keys, distance_threshold=8.0, min_component_size=3):
    """
    g: dataframe of one single protein
    residue_keys: list or set of predicted residues
    """
    residue_keys = list(residue_keys)
    if len(residue_keys) <= 1:
        return residue_keys

    sub = g[g["residue_key"].isin(residue_keys)].copy().reset_index(drop=True)
    if len(sub) <= 1:
        return residue_keys

    coords = sub[["x", "y", "z"]].values
    keys = sub["residue_key"].tolist()

    # Adjacency list
    adj = defaultdict(list)
    dmat = np.linalg.norm(coords[:, None, :] - coords[None, :, :], axis=-1)

    for i in range(len(sub)):
        for j in range(i + 1, len(sub)):
            if dmat[i, j] < distance_threshold:
                adj[i].append(j)
                adj[j].append(i)

    visited = set()
    components = []

    for i in range(len(sub)):
        if i in visited:
            continue

        q = deque([i])
        visited.add(i)
        comp = []

        while q:
            node = q.popleft()
            comp.append(node)
            for nb in adj[node]:
                if nb not in visited:
                    visited.add(nb)
                    q.append(nb)

        components.append(comp)

    # Consider only components that are large enough
    components = [c for c in components if len(c) >= min_component_size]

    if len(components) == 0:
        # Fallback: highest scoring residue is kept
        top_res = sub.sort_values("pred", ascending=False).iloc[0]["residue_key"]
        return [top_res]

    largest = max(components, key=len)
    return [keys[i] for i in largest]

In [49]:
# predictions

def predict_test(models, test_df, feature_cols):
    """
    Generates prediction scores for the test set by averaging the outputs
    of all trained fold models.

    Each model predicts a probability for every residue in test_df.
    The final score per residue is the mean probability across all models
    and is stored in the new column 'pred'.
    """
    X_test = test_df[feature_cols].copy()

    preds = np.zeros(len(test_df), dtype=float)

    for model in models:
        preds += model.predict_proba(X_test)[:, 1]

    preds /= len(models)

    test_df = test_df.copy()
    test_df["pred"] = preds

    return test_df

In [50]:
if __name__ == "__main__":
    # -------------------------
    # ADJUST PATHS
    # -------------------------

    # paths Laptop
    #TRAIN_CSV_PATH = "C:/Users/Lukas/Master_AI/2nd_Semester/KV_Structural_Bioinformatics/train.csv"
    #TRAIN_PDB_DIR = "C:/Users/Lukas/Master_AI/2nd_Semester/KV_Structural_Bioinformatics/train"
    #TEST_PDB_DIR = "C:/Users/Lukas/Master_AI/2nd_Semester/KV_Structural_Bioinformatics/test"

    # paths PC
    TRAIN_CSV_PATH = "C:/Users/stec/PycharmProjects/Master_AI/2nd_Semester/KV_Structural_Bioinformatics/train.csv"
    TRAIN_PDB_DIR = "C:/Users/stec/PycharmProjects/Master_AI/2nd_Semester/KV_Structural_Bioinformatics/train"
    TEST_PDB_DIR = "C:/Users/stec/PycharmProjects/Master_AI/2nd_Semester/KV_Structural_Bioinformatics/test"

    # -------------------------
    # LOAD LABELS
    # -------------------------
    print("Load train.csv ...")
    train_labels = load_train_labels(TRAIN_CSV_PATH)

    print()
    print("First rows of train.csv:")
    print(train_labels.head())

    print()
    print("Example binding_set for ID 0:")
    print(train_labels.loc[train_labels["id"] == "0", "binding_set"].iloc[0])

    # -------------------------
    # BUILD TRAINING DATAFRAME
    # -------------------------
    print()
    print("=" * 60)
    print("BUILD TRAIN DATAFRAME")
    print("=" * 60)
    
    train_df, missing_df = build_train_dataframe(
        TRAIN_PDB_DIR,
        train_labels,
        max_files=None,
        drop_bad_proteins=False,
        max_missing_fraction=0.30,
        max_missing_abs=5
    )

    print()
    print("Train-DataFrame Shape:")
    print(train_df.shape)

    print()
    print("First rows:")
    print(train_df.head())

    print()
    print("Distribution target:")
    print(train_df["target"].value_counts())

    print()
    print("Number of positive residues:")
    print(train_df["target"].sum())

    # -------------------------
    # RAM OPTIMIZATION
    # -------------------------
    train_df = train_df.drop(columns=[
        "n_labels_total",
        "n_missing_labels",
        "missing_fraction",
        "is_problematic_protein"
    ])

    print()
    print("Train-DataFrame after Cleanup:")
    print(train_df.shape)

    # -------------------------
    # CALCULATE FEATURES
    # -------------------------
    print()
    print("=" * 60)
    print("CALCULATE FEATURES")
    print("=" * 60)

    train_df = add_basic_aa_features(train_df)
    train_df = add_geometric_features(train_df)

    # -------------------------
    # FEATURE COULUMNS FOR THE MODEL
    # -------------------------
    feature_cols = [
        "aa",
        "chain_id",
        "is_hydrophobic",
        "is_polar",
        "is_positive",
        "is_negative",
        "is_aromatic",
        "dist_to_center",
        "dist_to_center_z",
        "n_neighbors_4",
        "n_neighbors_6",
        "n_neighbors_8",
        "n_neighbors_10",
        "n_neighbors_12",
        "min_knn_1",
        "mean_knn_3",
        "mean_knn_5",
        "mean_knn_10",
        "std_knn_5",
        "std_knn_10",
        "frac_hydrophobic_nbrs_8",
        "frac_polar_nbrs_8",
        "frac_positive_nbrs_8",
        "frac_negative_nbrs_8",
        "frac_aromatic_nbrs_8",
    ]

    # set categories for LightGBM
    train_df["aa"] = train_df["aa"].astype("category")
    train_df["chain_id"] = train_df["chain_id"].astype("category")

    # -------------------------
    # TRAIN MODEL
    # -------------------------
    print()
    print("=" * 60)
    print("MODEL TRAINING")
    print("=" * 60)

    models, train_df = train_lgbm(train_df, feature_cols, n_splits=5)

    print()
    print("OOF Predictions added:")
    print(train_df[["pdb_id", "residue_key", "target", "pred"]].head())

    # -------------------------
    # OPTIMIZE THRESHOLD FOR OFFICIAL METRIC
    # -------------------------
    print()
    print("=" * 60)
    print("THRESHOLD OPTIMIZATION (OFFICIAL METRIC)")
    print("=" * 60)

    best_thr_official, best_score_official = find_best_threshold_official_with_postprocessing(
        train_df,
        distance_threshold=8.0,
        min_component_size=3
    )

    print()
    print("Best Threshold for official Metric:")
    print(f"Threshold: {best_thr_official:.2f}")
    print(f"Official Jaccard: {best_score_official:.4f}")

    # -------------------------
    # BUILD TEST-DATAFRAME 
    # -------------------------
    print()
    print("=" * 60)
    print("BUILD TEST-DATAFRAME")
    print("=" * 60)

    test_df = build_test_dataframe(TEST_PDB_DIR, max_files= None)

    print("Test-DataFrame Shape:")
    print(test_df.shape)

    # -------------------------
    # TEST FEATURES
    # -------------------------
    print()
    print("=" * 60)
    print("TEST FEATURES")
    print("=" * 60)

    test_df = add_basic_aa_features(test_df)
    test_df = add_geometric_features(test_df)

    test_df["aa"] = test_df["aa"].astype("category")
    test_df["chain_id"] = test_df["chain_id"].astype("category")

    # -------------------------
    # TEST PREDICTION
    # -------------------------
    print()
    print("=" * 60)
    print("TEST PREDICTION")
    print("=" * 60)

    test_df = predict_test(models, test_df, feature_cols)

    print(test_df[["pdb_id", "residue_key", "pred"]].head())


    # -------------------------
    # CREATE SUBMISSION 
    # -------------------------
    print()
    print("=" * 60)
    print("CREATE SUBMISSION")
    print("=" * 60)

    submission_df = build_submission(
        test_df,
        threshold=best_thr_official,
        output_path="submission.csv",
        distance_threshold=8.0,
        min_component_size=3
    )

    print()
    print("DONE")
    print(f"Used Threshold: {best_thr_official:.2f}")
    print("Submission File: submission.csv")

    print(submission_df.shape)
    print(submission_df.head())
    print(submission_df.columns)

Load train.csv ...

First rows of train.csv:
  id                                              resid  \
0  0  A_24 A_25 A_26 A_27 A_28 A_29 A_30 A_31 A_32 A...   
1  1  A_14 A_15 A_16 A_28 A_30 A_31 A_54 A_87 A_89 A...   
2  2  A_249 A_255 A_259 A_268 A_270 A_277 A_278 A_27...   
3  3  C_142 C_143 C_144 C_146 C_147 C_148 C_507 C_50...   
4  4  A_162 A_163 A_164 A_201 A_204 A_206 A_207 A_22...   

                                         binding_set  
0  {A_402, A_111, A_404, A_321, A_249, A_43, A_10...  
1  {A_164, A_111, A_136, A_133, A_157, A_54, A_10...  
2  {A_470, A_472, A_329, A_292, A_289, A_350, A_3...  
3  {C_512, C_562, C_148, C_526, C_515, C_513, C_5...  
4  {A_164, A_328, A_299, A_308, A_333, A_206, A_3...  

Example binding_set for ID 0:
{'A_402', 'A_111', 'A_404', 'A_321', 'A_249', 'A_43', 'A_109', 'A_398', 'A_108', 'A_37', 'A_38', 'A_33', 'A_28', 'A_27', 'A_107', 'A_152', 'A_355', 'A_29', 'A_259', 'A_324', 'A_34', 'A_26', 'A_32', 'A_35', 'A_319', 'A_467', 'A_25', 'A_317'